# 1. Import the necessary libraries

In [3]:
# Adds the parent directory to sys.path
import sys
sys.path.append('../')

from mesh.core import *
from mesh.MESH_old import *
from runners import run_mesh, run_mesh_old, run_nsga2
from problems.benchmark_problems import get_problem

import contextlib
import cProfile
import io
import numpy as np
import timeit
import statistics
import pstats

# 2. Experiment Configuration

In [ ]:
#################### CUSTOMIZABLE ####################

# Number of processes to execute the experiments in parallel
workers = 2

# Objective function configuration (function name, number of objectives, number of decision variables)
experiments = [('zdt1', 2, 10, None), ('zdt2', 2, 10, None)]
# experiments = [
#     ('zdt1', 2, 10, None), ('zdt2', 2, 10, None), ('zdt3', 2, 10, None), ('zdt4', 2, 10, None), ('zdt6', 2, 10, None),
#     ('dtlz1', 3, 10, None), ('dtlz2', 3, 10, None), ('dtlz3', 3, 10, None), ('dtlz4', 3, 10, None), ('dtlz5', 3, 10, None), ('dtlz6', 3, 10, None), ('dtlz7', 3, 10, None),
#     ('wfg1', 3, 10, 6), ('wfg2', 3, 10, 6), ('wfg3', 3, 10, 6), ('wfg4', 3, 10, 6), ('wfg5', 3, 10, 6), ('wfg6', 3, 10, 6), ('wfg7', 3, 10, 6), ('wfg8', 3, 10, 6),
#     ('wfg9', 3, 10, 6)]

# Execution configuration
num_repeats = 5 # Number of runs for collecting statistics
max_fitness_eval = 15000 # Maximum fitness evaluations (not used if it is None)
population_size = 100 # Population size
random_state = None # Set a seed for random numbers (not used if it is None)

results_folder = f'../results' # Folder to store the results
tuning_folder = f'../hyperparams' # Folder to get the tuned parameters
######################################################

# 3. Auxiliar Functions

In [ ]:
# Function to get the fitness function configuration
def get_exp_problem(func_name, objective_dim, position_dim, wfg_k):
	func, position_min_value, position_max_value = get_problem(func_name, n_var=position_dim, n_obj=objective_dim, wfg_k=wfg_k)
	return func, objective_dim, position_dim, position_min_value, position_max_value

# Function to capture errors during parallel execution without stopping the other processes
def safe_run(run_function, params):
	try:
		f = io.StringIO()
		with contextlib.redirect_stdout(f):
			status = run_function(*params)
		return status, f.getvalue()
	except Exception as e:
		return f'Error: {str(e)}'

# 4. Benchmark

## 4.1 MESH

In [ ]:
#################### CUSTOMIZABLE ####################
# MESH fixed parameters
mesh_memory_size = population_size # Maximum number of particles in memory
mesh_global_best_type = [0] # 0 -> Sigma method (G1) | 1 -> Sigma method in fronts (G2)
mesh_dm_pool_type = [0] # 0 -> Sampling from memory (S1) | 1 -> Sampling from population (S2) | 2 -> Sampling from memory and population (S3)
mesh_differential_evolution_type = [0] # 0 -> DE\rand\1\Bin (D1) | 1 -> DE\rand\2\Bin (D2) | 2 -> DE/Best/1/Bin (D3) | 3 -> DE/Current-to-best/1/Bin (D4) | 4 -> DE/Current-to-rand/1/Bin (D5)


# MESH tunable parameters
# OBS: The function "run_mesh" and "run_mesh_old" will select automatically the tuned parameters if they exists ("hyperparams" folder generated by "fine_tuning.ipynb")
mesh_communication_probability = 0.8 # Communication probability
mesh_mutation_rate = 0.9 # Mutation rate
mesh_personal_guide_array_size = 1 # Number of personal guides
######################################################

In [ ]:
# Set the list of parameters
params_list = [
  	[(F'MESH_G{gb_type+1}S{pool_type+1}D{de_type+1}_{exp[0]}_{exp[1]}_{exp[2]}', results_folder, tuning_folder, 1, max_fitness_eval, population_size, random_state),
	 get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
	 (mesh_memory_size, gb_type, pool_type, de_type),
	 (mesh_communication_probability, mesh_mutation_rate, mesh_personal_guide_array_size)]
     for exp in experiments
     for gb_type in mesh_global_best_type
	 for pool_type in mesh_dm_pool_type
	 for de_type in mesh_differential_evolution_type
]

# Execute in parallel
Parallel(n_jobs=workers)(delayed(safe_run)(run_mesh, params) for params in params_list)

In [ ]:


objective_dim = 5
position_dim = 10
max_iterations = None
max_fitness_eval = 10000
population_size = 500

random_state = 42

position_min_value = np.array([0]*position_dim)
position_max_value = np.array([1]*position_dim)
num_final_solutions = population_size
memory_size = population_size
communication_probability = 0.7
mutation_rate = 0.9
personal_guide_array_size = 3
global_best_attribution_type = 0
dm_pool_type = 0
dm_operation_type = 0
crowding_distance_type = 0
optimization_type = [False]*objective_dim
def generate_objective_function(objective_dim):
    def objective_function(position):
        position = np.array(position)
        objectives = []
        for i in range(objective_dim):
            # Alterna entre diferentes padrões para criar trade-offs
            if i % 3 == 0:
                # Função do tipo esfera com deslocamento
                obj = np.sum((position - (i + 1))**2)
            elif i % 3 == 1:
                # Função baseada em ondas (oscilação não-linear)
                obj = np.sum(np.sin(position * (i + 1))**2)
            else:
                # Função mista: produto entre posição e índice do objetivo
                obj = np.prod(1 + position / (i + 1))
            # Normaliza cada objetivo em relação ao índice
            obj /= (i + 1) * 10
            objectives.append(obj)
        return np.array(objectives)
    return objective_function
func = generate_objective_function(objective_dim)

def run_new():
    params = MeshParameters(objective_dim,
                             position_dim, position_min_value, position_max_value, 
                             population_size, memory_size,
                             global_best_attribution_type, dm_pool_type, dm_operation_type,
                             communication_probability, mutation_rate,
                             max_gen=max_iterations, max_fit_eval=max_fitness_eval,
                             max_personal_guides=personal_guide_array_size,
                             random_state=random_state)
    ##########################################################
    #experiment_name = 'zdt2'
    #func = get_problem(experiment_name, n_var=position_dim).evaluate
    ##########################################################
    new_mesh = Mesh(params, func)
    new_mesh.run()

def run_old():
    params_old = MESH_Params_old(objective_dim,
                                [False]*objective_dim,
                                max_iterations,
                                max_fitness_eval,
                                position_dim,
                                position_max_value, position_min_value,
                                population_size,memory_size,
                                0,
                                global_best_attribution_type,
                                dm_operation_type,
                                dm_pool_type,
                                crowding_distance_type,
                                communication_probability,
                                mutation_rate,
                                personal_guide_array_size)
    old_mesh = MESH_old(params_old, func)
    old_mesh.run()


# print("Estatísticas para algoritmo original:")
# times = timeit.repeat("run_old()", globals=globals(), repeat=30, number=1)
# # Estatísticas
# mean_time = statistics.mean(times)
# std_dev_time = statistics.stdev(times)
# min_time = min(times)
# max_time = max(times)
# print(f"Tempo médio: {mean_time:.6f} segundos")
# print(f"Desvio padrão: {std_dev_time:.6f} segundos")
# print(f"Tempo mínimo: {min_time:.6f} segundos")
# print(f"Tempo máximo: {max_time:.6f} segundos\n\n")


print("Estatísticas para algoritmo otimizado:")
times = timeit.repeat("run_new()", globals=globals(), repeat=30, number=1)
# Estatísticas
mean_time = statistics.mean(times)
std_dev_time = statistics.stdev(times)
min_time = min(times)
max_time = max(times)
print(f"Tempo médio: {mean_time:.6f} segundos")
print(f"Desvio padrão: {std_dev_time:.6f} segundos")
print(f"Tempo mínimo: {min_time:.6f} segundos")
print(f"Tempo máximo: {max_time:.6f} segundos")

# cProfile.run('run_new()', sort='time', filename="profile.prof")
# stats = pstats.Stats('profile.prof')
# stats.strip_dirs()
# stats.sort_stats('time')
# stats.print_stats(10)